# Notebook 07 — Full RAVDESS Evaluation on Colab

This runner downloads RAVDESS, creates the complete prediction manifest, evaluates deletion-faithfulness and class-specificity for every correctly classified utterance, stores results in Google Drive, and downloads one ZIP. Use a GPU runtime.

In [1]:
# SETUP: Drive, repository, dependencies, LeGrad, RAVDESS and predictions
from pathlib import Path
import os, shutil, subprocess, sys, urllib.request, zipfile

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/MateusWiteck/gradient_based_speach_xai.git'
PROJECT_DIR = Path('/content/gradient_based_speach_xai')
RAVDESS_DIR = Path('/content/data/ravdess')
RAVDESS_ZIP = RAVDESS_DIR / 'Audio_Speech_Actors_01-24.zip'
RAVDESS_URL = 'https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1'
OUTPUT_ROOT = Path('/content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full')
REPORT_METHODS = [
    'level3',
    'legrad_final_score_relu_attention_gradient_mean_layers_source_tokens',
    'legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens',
    'legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens',
    'legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens',
]
METHODS_ARG = ','.join(REPORT_METHODS)

def run(command, cwd=None):
    command = [str(part) for part in command]
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})

run(['nvidia-smi'])
if PROJECT_DIR.exists() and not (PROJECT_DIR / '.git').exists():
    shutil.rmtree(PROJECT_DIR)
if not PROJECT_DIR.exists():
    run(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, PROJECT_DIR])
else:
    run(['git', 'fetch', 'origin', 'main'], cwd=PROJECT_DIR)
    run(['git', 'checkout', '-B', 'main', 'origin/main'], cwd=PROJECT_DIR)
run(['git', 'submodule', 'update', '--init', '--recursive', 'third_party/LeGrad'], cwd=PROJECT_DIR)
run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-r', PROJECT_DIR / 'requirements.txt'], cwd=PROJECT_DIR)
run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print('GPU:', torch.cuda.get_device_name(0))"], cwd=PROJECT_DIR)

RAVDESS_DIR.mkdir(parents=True, exist_ok=True)
ravdess_root = next((p.parent for p in RAVDESS_DIR.rglob('Actor_01') if (p.parent / 'Actor_24').is_dir()), None)
if ravdess_root is None:
    if not RAVDESS_ZIP.exists() or not zipfile.is_zipfile(RAVDESS_ZIP):
        if RAVDESS_ZIP.exists(): RAVDESS_ZIP.unlink()
        print('Downloading RAVDESS...', flush=True)
        urllib.request.urlretrieve(RAVDESS_URL, RAVDESS_ZIP)
    with zipfile.ZipFile(RAVDESS_ZIP) as archive:
        destination = RAVDESS_DIR.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    ravdess_root = next((p.parent for p in RAVDESS_DIR.rglob('Actor_01') if (p.parent / 'Actor_24').is_dir()), None)
if ravdess_root is None:
    raise FileNotFoundError('Could not locate Actor_01 through Actor_24 after extraction.')
audio_count = len(list(ravdess_root.glob('Actor_*/*.wav')))
if audio_count < 670:
    raise RuntimeError(f'Incomplete RAVDESS extraction: found only {audio_count} WAV files.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
prediction_root = OUTPUT_ROOT / 'predictions'
run([sys.executable, '-u', 'scripts/evaluate_ravdess.py', '--ravdess-root', ravdess_root, '--batch-size', '16', '--device', 'cuda', '--output-root', prediction_root], cwd=PROJECT_DIR)
prediction_runs = sorted(prediction_root.glob('ravdess_external_eval_*'), key=lambda p: p.stat().st_mtime)
if not prediction_runs: raise RuntimeError('RAVDESS prediction run was not created.')
PREDICTIONS_CSV = prediction_runs[-1] / 'predictions.csv'
print('RAVDESS root:', ravdess_root)
print('Audio files:', audio_count)
print('Predictions:', PREDICTIONS_CSV)
print('Output root:', OUTPUT_ROOT)

Mounted at /content/drive
+ nvidia-smi
+ git clone --branch main --single-branch https://github.com/MateusWiteck/gradient_based_speach_xai.git /content/gradient_based_speach_xai
+ git submodule update --init --recursive third_party/LeGrad
+ /usr/bin/python3 -m pip install --upgrade pip wheel
+ /usr/bin/python3 -m pip install -r /content/gradient_based_speach_xai/requirements.txt
+ /usr/bin/python3 -c import torch; assert torch.cuda.is_available(); print('GPU:', torch.cuda.get_device_name(0))
+ /usr/bin/python3 -u scripts/evaluate_ravdess.py --ravdess-root /content/data/ravdess --batch-size 16 --device cuda --output-root /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/predictions
RAVDESS root: /content/data/ravdess
Audio files: 1440
Predictions: /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/predictions/ravdess_external_eval_20260712_100936_162779/predictions.csv
Output root: /content/drive/MyDrive/gradient_based_speech_xai_ou

In [2]:
# FULL DELETION-FAITHFULNESS EVALUATION
deletion_root = OUTPUT_ROOT / 'deletion_faithfulness'
run([
    sys.executable, '-u', 'scripts/evaluate_deletion_faithfulness.py',
    '--predictions-csv', PREDICTIONS_CSV, '--dataset-name', 'ravdess',
    '--max-examples', '1000000', '--fractions', '0.05,0.1,0.2,0.3',
    '--random-trials', '3', '--modes', METHODS_ARG, '--device', 'cuda',
    '--output-root', deletion_root,
], cwd=PROJECT_DIR)
print('Deletion-faithfulness complete:', deletion_root)

+ /usr/bin/python3 -u scripts/evaluate_deletion_faithfulness.py --predictions-csv /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/predictions/ravdess_external_eval_20260712_100936_162779/predictions.csv --dataset-name ravdess --max-examples 1000000 --fractions 0.05,0.1,0.2,0.3 --random-trials 3 --modes level3,legrad_final_score_relu_attention_gradient_mean_layers_source_tokens,legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens,legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens,legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens --device cuda --output-root /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/deletion_faithfulness
Deletion-faithfulness complete: /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/deletion_faithfulness


In [3]:
# FULL CLASS-SPECIFICITY EVALUATION
specificity_root = OUTPUT_ROOT / 'class_specificity'
run([
    sys.executable, '-u', 'scripts/evaluate_class_specificity.py',
    '--predictions-csv', PREDICTIONS_CSV, '--dataset-name', 'ravdess',
    '--max-examples', '1000000', '--modes', METHODS_ARG, '--device', 'cuda',
    '--output-root', specificity_root,
], cwd=PROJECT_DIR)
print('Class-specificity complete:', specificity_root)

+ /usr/bin/python3 -u scripts/evaluate_class_specificity.py --predictions-csv /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/predictions/ravdess_external_eval_20260712_100936_162779/predictions.csv --dataset-name ravdess --max-examples 1000000 --modes level3,legrad_final_score_relu_attention_gradient_mean_layers_source_tokens,legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens,legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens,legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens --device cuda --output-root /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/class_specificity
Class-specificity complete: /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full/class_specificity


In [4]:
# ZIP ALL NOTEBOOK 07 RESULTS AND DOWNLOAD THEM
from datetime import datetime
from google.colab import files

required = [OUTPUT_ROOT / 'predictions', OUTPUT_ROOT / 'deletion_faithfulness', OUTPUT_ROOT / 'class_specificity']
missing = [str(path) for path in required if not path.is_dir() or not any(path.rglob('*'))]
if missing: raise FileNotFoundError('Missing result directories: ' + ', '.join(missing))
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_base = Path(f'/content/notebook_07_ravdess_results_{timestamp}')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name))
print('Results:', OUTPUT_ROOT)
print('ZIP:', archive_path)
print(f'Size: {archive_path.stat().st_size / (1024**2):.1f} MB')
files.download(str(archive_path))

Results: /content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_ravdess_full
ZIP: /content/notebook_07_ravdess_results_20260712_102249.zip
Size: 1.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>